# SQL- union is similar to Python-concat where vertical addition is done
# SQL-Self Join is similar to Python-merge where horizontal addition is done based on an unique key column

In [1]:
import pandas as pd
import numpy as np

In [2]:
old=pd.read_excel('old_file.xlsx')
old

,Employee_ID,Name,Department,Approver_ID,Threshold
0,101,Arun,Finance,A001,10000
1,102,Bhavna,HR,A002,15000
2,103,Chirag,IT,A003,12000
3,104,Deepa,Finance,A004,13000
4,105,Esha,Sales,A005,16000
5,106,Farhan,IT,A006,11000
6,107,Gita,HR,A007,18000
7,108,Hari,Finance,A008,12500


In [39]:
len(old)

8

In [3]:
new=pd.read_excel('new_file.xlsx')
new

,Employee_ID,Name,Department,Approver_ID,Threshold
0,101,Arun,Finance,A001,10000
1,102,Bhavna,HR,A009,15000
2,103,Chirag,IT,A003,15000
3,104,Deepa,Finance,A004,13000
4,105,Esha,Marketing,A005,20000
5,106,Farhan,IT,A010,11000
6,109,Isha,Sales,A011,17000
7,110,Jatin,Finance,A012,14000


In [40]:
len(new)

8

In [26]:
merged=pd.merge(old,new, on="Employee_ID",how="outer",suffixes=("_old", "_new"),indicator=True)
merged

,Employee_ID,Name_old,Department_old,Approver_ID_old,Threshold_old,source_old,Name_new,Department_new,Approver_ID_new,Threshold_new,source_new,_merge
0,101,Arun,Finance,A001,10000.0,old,Arun,Finance,A001,10000.0,new,both
1,102,Bhavna,HR,A002,15000.0,old,Bhavna,HR,A009,15000.0,new,both
2,103,Chirag,IT,A003,12000.0,old,Chirag,IT,A003,15000.0,new,both
3,104,Deepa,Finance,A004,13000.0,old,Deepa,Finance,A004,13000.0,new,both
4,105,Esha,Sales,A005,16000.0,old,Esha,Marketing,A005,20000.0,new,both
5,106,Farhan,IT,A006,11000.0,old,Farhan,IT,A010,11000.0,new,both
6,107,Gita,HR,A007,18000.0,old,NaN,NaN,NaN,NaN,NaN,left_only
7,108,Hari,Finance,A008,12500.0,old,NaN,NaN,NaN,NaN,NaN,left_only
8,109,NaN,NaN,NaN,NaN,NaN,Isha,Sales,A011,17000.0,new,right_only
9,110,NaN,NaN,NaN,NaN,NaN,Jatin,Finance,A012,14000.0,new,right_only


In [24]:
len(merged) # 12 entries repeated so combined 6, 2 entries to new addition and 2 entries from old file, total 10

10

In [27]:
Added=merged[merged['_merge']=='right_only']
Added

,Employee_ID,Name_old,Department_old,Approver_ID_old,Threshold_old,source_old,Name_new,Department_new,Approver_ID_new,Threshold_new,source_new,_merge
8,109,NaN,NaN,NaN,NaN,NaN,Isha,Sales,A011,17000.0,new,right_only
9,110,NaN,NaN,NaN,NaN,NaN,Jatin,Finance,A012,14000.0,new,right_only


In [28]:
Removed=merged[merged['_merge']=='left_only']
Removed

,Employee_ID,Name_old,Department_old,Approver_ID_old,Threshold_old,source_old,Name_new,Department_new,Approver_ID_new,Threshold_new,source_new,_merge
6,107,Gita,HR,A007,18000.0,old,NaN,NaN,NaN,NaN,NaN,left_only
7,108,Hari,Finance,A008,12500.0,old,NaN,NaN,NaN,NaN,NaN,left_only


In [30]:
common=merged[merged['_merge']=='both'] # it includes matching and non matching both entries from 2 files i.e. 12 length
common

,Employee_ID,Name_old,Department_old,Approver_ID_old,Threshold_old,source_old,Name_new,Department_new,Approver_ID_new,Threshold_new,source_new,_merge
0,101,Arun,Finance,A001,10000.0,old,Arun,Finance,A001,10000.0,new,both
1,102,Bhavna,HR,A002,15000.0,old,Bhavna,HR,A009,15000.0,new,both
2,103,Chirag,IT,A003,12000.0,old,Chirag,IT,A003,15000.0,new,both
3,104,Deepa,Finance,A004,13000.0,old,Deepa,Finance,A004,13000.0,new,both
4,105,Esha,Sales,A005,16000.0,old,Esha,Marketing,A005,20000.0,new,both
5,106,Farhan,IT,A006,11000.0,old,Farhan,IT,A010,11000.0,new,both


In [36]:
cols_to_compare = ["Approver_ID", "Threshold", "Department"]
changes = [] # empty list to collect diffs
for col in cols_to_compare:
    diff = common[common[f"{col}_old"] != common[f"{col}_new"]] #check entries where columns not matching
    if not diff.empty: # if any column found with mismatch from the above step
        diff_summary = diff[["Employee_ID", f"{col}_old", f"{col}_new"]].copy()
        diff_summary["Changed_Column"] = col
        changes.append(diff_summary)
        
# Combine all differences
if changes:
    changes_df = pd.concat(changes,ignore_index=True)
    print("Differences found:")
    print(changes_df)
else:
    print("No changes found.")

Differences found:
   Employee_ID Approver_ID_old Approver_ID_new Changed_Column  Threshold_old  \
0          102            A002            A009    Approver_ID            NaN   
1          106            A006            A010    Approver_ID            NaN   
2          103             NaN             NaN      Threshold        12000.0   
3          105             NaN             NaN      Threshold        16000.0   
4          105             NaN             NaN     Department            NaN   

   Threshold_new Department_old Department_new  
0            NaN            NaN            NaN  
1            NaN            NaN            NaN  
2        15000.0            NaN            NaN  
3        20000.0            NaN            NaN  
4            NaN          Sales      Marketing  


In [38]:
differences_found_summary=pd.DataFrame(changes_df)
differences_found_summary

,Employee_ID,Approver_ID_old,Approver_ID_new,Changed_Column,Threshold_old,Threshold_new,Department_old,Department_new
0,102,A002,A009,Approver_ID,NaN,NaN,NaN,NaN
1,106,A006,A010,Approver_ID,NaN,NaN,NaN,NaN
2,103,NaN,NaN,Threshold,12000.0,15000.0,NaN,NaN
3,105,NaN,NaN,Threshold,16000.0,20000.0,NaN,NaN
4,105,NaN,NaN,Department,NaN,NaN,Sales,Marketing
